In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

# ============================================================
# Device Setup
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print("CUDA version:", torch.version.cuda)
    print("GPU:", torch.cuda.get_device_name(0))

# ============================================================
# DIAGNOSIS: Current RL Issues
# ============================================================
print("\n" + "="*60)
print("DIAGNOSIS OF CURRENT RL ISSUES")
print("="*60)

print("="*60 + "\n")

Using device: cpu

DIAGNOSIS OF CURRENT RL ISSUES



In [ ]:
# ============================================================
# 1. 1D Diffusion Solver (Generate Data)
# ============================================================
def diffusion_solver(u0, nu=0.01, dx=1/128, dt=1e-4, n_steps=50000):
    """Explicit finite-difference diffusion solver (periodic BCs)."""
    u = u0.copy()
    traj = [u.copy()]
    for _ in range(n_steps):
        lap = (np.roll(u, -1) - 2*u + np.roll(u, 1)) / dx**2
        u = u + nu * dt * lap
        traj.append(u.copy())
    return np.stack(traj)

# Generate 1D data
Nx = 256
dx = 1/Nx
dt0 = 1e-4
nu = 0.01
n_steps = 50000

x = np.linspace(0, 1, Nx, endpoint=False)
u0 = np.sin(2*np.pi*x)

print("Generating 1D diffusion data...")
data = diffusion_solver(u0, nu, dx, dt0, n_steps)
print(f"Data shape: {data.shape}")

plt.figure(figsize=(10, 5))

plt.plot(x, data[0], linewidth=2, label="Initial (t = 0)")
plt.plot(x, data[n_steps//2], linewidth=2, label=f"Midpoint (t = {n_steps//2})")
plt.plot(x, data[-1], linewidth=2, label=f"Final (t = {n_steps})")

plt.title("Diffusion: Initial, Midpoint, and Final States")
plt.xlabel("x")
plt.ylabel("u(x)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Train/test split
T_train = 25000
u_train = torch.tensor(data[:T_train], dtype=torch.float32, device=device)
u_test_true = torch.tensor(data[T_train:], dtype=torch.float32, device=device)


In [ ]:
# Train/test split
T_train = 25000
u_train = torch.tensor(data[:T_train], dtype=torch.float32, device=device)
u_test_true = torch.tensor(data[T_train:], dtype=torch.float32, device=device)

# ============================================================
# 2. Load Pretrained Neural Stepper Model
# ============================================================
class MLPStepper(nn.Module):
    def __init__(self, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    
    def forward(self, u_left, u_center, u_right, dt_norm):
        """Takes individual spatial points: (batch,), (batch,), (batch,), (batch,)"""
        inp = torch.stack([u_left, u_center, u_right, dt_norm], dim=-1)
        du = self.net(inp).squeeze(-1)
        return u_center + 0.005 * du

stepper = MLPStepper(hidden=128).to(device)

# Load pretrained stepper
print("Loading pretrained stepper from neural_diffusion_stepper2.pt...")
stepper.load_state_dict(torch.load('neural_diffusion_stepper2.pt', map_location=device, weights_only=False))
stepper.eval()  # Set to evaluation mode
print("Stepper loaded successfully!")


In [ ]:
# ============================================================
# 3. IMPROVED PPO Actor-Critic (Full Field + Extended dt Range)
# ============================================================
# State: Full u field (256 dims) + summary statistics
# Actions: Discrete {1, 2, 4, 8, 9} for dt multiplier (extended to test generalization)

def extract_state_features(u_field, include_full=True):
    """
    Extract state features from field u.
    Input: u_field (batch, Nx) or (Nx,)
    Output: (batch, state_dim) - full field only
    """
    if u_field.dim() == 1:
        u_field = u_field.unsqueeze(0)
    
    # Normalize full field (zero mean, unit std per sample)
    mean_u = u_field.mean(dim=-1, keepdim=True)  # (batch, 1)
    std_u = u_field.std(dim=-1, keepdim=True)     # (batch, 1)
    u_normalized = (u_field - mean_u) / (std_u + 1e-8)
    
    return u_normalized.squeeze(0) if u_normalized.shape[0] == 1 else u_normalized

class ImprovedActorCritic(nn.Module):
    def __init__(self, state_dim=256, n_actions=6, hidden=128):  # 256 (field only)
        super().__init__()
        self.n_actions = n_actions
        self.dt_choices = [1, 2, 4, 6, 8, 10]  # Includes dt=10
        self.state_dim = state_dim
        
        # Field encoder: process full field
        self.field_encoder = nn.Sequential(
            nn.Linear(256, hidden),  # Encode full field
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU()
        )
        
        # Actor: outputs logits for discrete actions
        self.actor = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )
        
        # Critic: outputs value estimate (larger network for better value estimation)
        self.critic = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1)
        )
    
    def forward(self, state_features):
        """
        state_features: (batch, 256) - u_field only
        """
        # Encode field
        feat = self.field_encoder(state_features)  # (batch, hidden)
        
        logits = self.actor(feat)  # (batch, n_actions)
        value = self.critic(feat)  # (batch, 1)
        return logits, value
    
    def sample_action(self, state_features):
        logits, value = self.forward(state_features)
        dist = torch.distributions.Categorical(logits=logits)
        action_idx = dist.sample()
        log_prob = dist.log_prob(action_idx)
        dt_mult = self.dt_choices[action_idx.item()]
        return dt_mult, action_idx, log_prob, value
    
    def evaluate(self, state_features, action_idx):
        logits, value = self.forward(state_features)
        dist = torch.distributions.Categorical(logits=logits)
        log_prob = dist.log_prob(action_idx)
        entropy = dist.entropy()
        return log_prob, value, entropy

agent = ImprovedActorCritic(state_dim=256, n_actions=6, hidden=128).to(device)
agent_opt = optim.Adam(agent.parameters(), lr=1e-3)

print("Improved Agent Architecture:")
print(f"  - State dim: 256 (full field only)")
print(f"  - Action space: Discrete {agent.dt_choices}")
print(f"  - Hidden units: 128")
print(f"  - Learning rate: 1e-3")
print(f"  - Field encoder: 256 -> 128 -> 128")


In [ ]:
# ============================================================
# 4. SIMPLIFIED PPO Training Loop with Better 



# ============================================================
def compute_gae(rewards, values, gamma=0.99, lam=0.95):
    """Generalized Advantage Estimation."""
    advantages = []
    gae = 0
    next_value = 0
    # Convert rewards to tensor if needed
    if isinstance(rewards, list):
        rewards = torch.tensor(rewards, dtype=torch.float32, device=device)
    # Ensure values is a tensor
    if isinstance(values, list):
        values = torch.tensor(values, dtype=torch.float32, device=device)
    # Ensure values is 1D
    if values.dim() > 1:
        values = values.squeeze()
    
    for t in reversed(range(len(rewards))):
        reward_t = rewards[t].item() if isinstance(rewards[t], torch.Tensor) else rewards[t]
        value_t = values[t].item() if isinstance(values[t], torch.Tensor) else values[t]
        delta = reward_t + gamma * next_value - value_t
        gae = delta + gamma * lam * gae
        advantages.insert(0, gae)
        next_value = value_t
    return torch.tensor(advantages, dtype=torch.float32, device=device)

def ppo_update(batch_states, batch_action_indices, batch_old_log_probs, 
               batch_rewards, batch_values, epsilon=0.2, n_epochs=5, 
               value_clip_epsilon=0.2, critic_coef=0.5):
    """PPO update with stabilized critic loss."""
    advantages = compute_gae(batch_rewards, batch_values)
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    
    # Compute returns with detached old values
    returns = advantages + batch_values.detach()
    
    # Normalize returns for stability
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    
    old_values = batch_values.detach()
    
    actor_losses = []
    critic_losses = []
    
    for epoch in range(n_epochs):
        log_probs, values, entropy = agent.evaluate(batch_states, batch_action_indices)
        values = values.squeeze()
        
        # Actor loss (clipped)
        ratio = torch.exp(log_probs - batch_old_log_probs)
        surr1 = ratio * advantages
        surr2 = torch.clamp(ratio, 1-epsilon, 1+epsilon) * advantages
        actor_loss = -torch.min(surr1, surr2).mean()
        
        # Critic loss with clipping (PPO2 style) - prevents large value updates
        value_clipped = old_values + torch.clamp(values - old_values, -value_clip_epsilon, value_clip_epsilon)
        value_loss1 = (values - returns) ** 2
        value_loss2 = (value_clipped - returns) ** 2
        critic_loss = 0.5 * torch.max(value_loss1, value_loss2).mean()
        
        entropy_loss = -0.01 * entropy.mean()
        
        total_loss = actor_loss + critic_coef * critic_loss + entropy_loss
        
        agent_opt.zero_grad()
        total_loss.backward()
        nn.utils.clip_grad_norm_(agent.parameters(), 0.5)
        agent_opt.step()
        
        actor_losses.append(actor_loss.item())
        critic_losses.append(critic_loss.item())
    
    return np.mean(actor_losses), np.mean(critic_losses)

print("\n" + "="*60)
print("Training IMPROVED PPO Agent")
print("="*60)
print("""
REWARD FUNCTION (TUNED):
  reward = -log(error + ε) + α * dt_mult - β * (dt_mult == max_dt)
  
  Where:
  - error = MSE between predicted and true state
  - ε = 1e-10 (small constant to avoid log(0))
  - α = 0.05 (speed bonus coefficient - REDUCED to encourage adaptation)
  - β = 0.02 (penalty for always choosing max dt)
  
  This reward:
  - Penalizes large errors (log scale is more sensitive)
  - Rewards using larger dt (but less aggressively)
  - Penalizes always choosing max dt to encourage adaptation
  - Provides better gradient signal
""")
print("="*60 + "\n")

n_episodes = 100
episode_rewards = []
episode_avg_dt = []
episode_avg_error = []
episode_dt_distribution = []  # Track distribution of dt choices
episode_entropy = []  # Track policy entropy (measure of exploration)
episode_critic_loss = []  # Track critic loss

# Reward parameters
epsilon_error = 1e-10  # Small constant to avoid log(0)
speed_bonus = 0.01     # REDUCED: Bonus per dt unit (was 0.1)
max_dt_penalty =0   # NEW: Penalty for always choosing max dt
max_dt = 10 # Maximum dt value

for episode in range(n_episodes):
    t = 0
    trajectory_states = []      # State features (mean, std, grad_norm)reward 
    trajectory_action_indices = []  # Action indices (0, 1, 2, 3)
    trajectory_old_log_probs = []
    trajectory_rewards = []
    trajectory_values = []
    
    u_current = u_train[t]
    total_reward = 0
    steps_taken = 0
    total_dt = 0
    total_error = 0
    
    dt_choices_episode = []  # Track dt choices in this episode
    
    while t < T_train - max_dt:
        # Extract state features (full field + summary stats)
        state_features = extract_state_features(u_current, include_full=True).unsqueeze(0).to(device)
        
        # Agent chooses dt (discrete action)
        with torch.no_grad():
            dt_mult, action_idx, log_prob, value = agent.sample_action(state_features)
        
        if t + dt_mult >= T_train:
            break
        
        dt_choices_episode.append(dt_mult)
        trajectory_states.append(state_features.clone())
        trajectory_action_indices.append(action_idx.clone())
        trajectory_old_log_probs.append(log_prob.clone())
        trajectory_values.append(value.clone())
        
        # Stepper predicts next state point-wise
        u_t = u_current.to(device)
        u_left = torch.roll(u_t, +1)
        u_right = torch.roll(u_t, -1)
        dt_norm = torch.full_like(u_t, float(dt_mult))
        
        with torch.no_grad():
            u_pred = stepper(u_left, u_t, u_right, dt_norm)
        
        # Get ground truth
        u_true = u_train[t + dt_mult].to(device)
        
        # TUNED REWARD: -log(error) + speed_bonus * dt - penalty for max_dt
        error = torch.mean((u_pred - u_true) ** 2).item()
        accuracy_penalty = -np.log(error + epsilon_error)  # Negative log error
        speed_reward = speed_bonus * dt_mult  # Bonus for using larger dt (reduced)
        max_dt_penalty_term = max_dt_penalty if dt_mult == max_dt else 0  # Penalty for always choosing max
        reward = accuracy_penalty + speed_reward  
        
        trajectory_rewards.append(reward)
        total_reward += reward
        total_dt += dt_mult
        total_error += error
        steps_taken += 1
        
        # Move to next state
        u_current = u_true
        t += dt_mult
    
    # PPO update
    if len(trajectory_rewards) > 0:
        batch_states = torch.cat(trajectory_states, dim=0)
        batch_action_indices = torch.cat(trajectory_action_indices, dim=0)
        batch_old_log_probs = torch.cat(trajectory_old_log_probs, dim=0)
        batch_values = torch.cat(trajectory_values, dim=0).squeeze()
        
        # Compute policy entropy for this batch
        with torch.no_grad():
            _, _, entropy_batch = agent.evaluate(batch_states, batch_action_indices)
            avg_entropy = entropy_batch.mean().item()
        
        actor_loss, critic_loss = ppo_update(
            batch_states, batch_action_indices, batch_old_log_probs,
            trajectory_rewards, batch_values,
            value_clip_epsilon=0.2,  # Clip value updates
            critic_coef=0.5  # Critic loss coefficient
        )
        
        episode_rewards.append(total_reward / steps_taken)
        episode_avg_dt.append(total_dt / steps_taken)
        episode_avg_error.append(total_error / steps_taken)
        episode_entropy.append(avg_entropy)
        episode_critic_loss.append(critic_loss)
        
        # Track dt distribution
        dt_dist = {dt: dt_choices_episode.count(dt) for dt in agent.dt_choices}
        episode_dt_distribution.append(dt_dist)
        
        if (episode + 1) % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            avg_dt = np.mean(episode_avg_dt[-10:])
            avg_error = np.mean(episode_avg_error[-10:])
            avg_ent = np.mean(episode_entropy[-10:])
            
            # Compute dt distribution over last 10 episodes
            recent_dt_dist = {}
            for dt in agent.dt_choices:
                recent_dt_dist[dt] = np.mean([episode_dt_distribution[i].get(dt, 0) 
                                             for i in range(max(0, episode-9), episode+1)])
            
            dt_dist_str = ", ".join([f"dt{dt}:{recent_dt_dist[dt]:.1f}" for dt in agent.dt_choices])
            
            print(f"Episode {episode+1}/{n_episodes} - "
                  f"Reward: {avg_reward:.2f}, "
                  f"Avg dt: {avg_dt:.2f}, "
                  f"Avg error: {avg_error:.2e}, "
                  f"Entropy: {avg_ent:.3f}, "
                  f"Steps: {steps_taken}")
            print(f"  dt distribution: {dt_dist_str}")
            print(f"  Actor Loss: {actor_loss:.4e}, Critic Loss: {critic_loss:.4e}")


In [ ]:
# ============================================================
# 5. Plotting Results
# ============================================================
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(episode_rewards)
plt.xlabel("Episode")
plt.ylabel("Average Reward")
plt.title("PPO Training Rewards")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(x, data[0], label="Initial", linewidth=2)
plt.plot(x, data[-1], label="Final", linewidth=2)
plt.xlabel("x")
plt.ylabel("u(x)")
plt.title("Diffusion: Initial vs Final")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print("\nTraining complete!")

In [ ]:
# ============================================================
# 5. Analyze Learned dt Strategy
# ============================================================
print("\n" + "="*60)
print("Analyzing Learned dt Strategy")
print("="*60)

# Ensure max_dt is defined (in case cell is run independently)
if 'max_dt' not in globals():
    max_dt = 10  # Default value

agent.eval()
dt_choices = []
field_states = []
state_features_list = []
t = 0
u_current = u_test_true[0]

while t < len(u_test_true) - max_dt:
    # Extract state features (full field + summary stats)
    state_features = extract_state_features(u_current, include_full=True).unsqueeze(0).to(device)
    
    with torch.no_grad():
        dt_mult, action_idx, _, _ = agent.sample_action(state_features)
    
    dt_choices.append(dt_mult)
    field_states.append(u_current.clone())
    state_features_list.append(state_features.squeeze().cpu().numpy())
    
    if t + dt_mult >= len(u_test_true):
        break
    
    u_current = u_test_true[t + dt_mult]
    t += dt_mult

print(f"Total steps in rollout: {len(dt_choices)}")
print(f"Mean dt: {np.mean(dt_choices):.2f}")
print(f"Min dt: {np.min(dt_choices)}, Max dt: {np.max(dt_choices)}")
print(f"Std dt: {np.std(dt_choices):.2f}")

# ============================================================
# 6. Plotting Results
# ============================================================
fig = plt.figure(figsize=(16, 10))

# Training rewards
ax1 = plt.subplot(2, 3, 1)
ax1.plot(episode_rewards)
ax1.set_xlabel("Episode")
ax1.set_ylabel("Average Reward")
ax1.set_title("PPO Training Rewards")
ax1.grid(True)

# Diffusion evolution
ax2 = plt.subplot(2, 3, 2)
ax2.plot(x, data[0], label="Initial", linewidth=2)
ax2.plot(x, data[-1], label="Final", linewidth=2)
ax2.set_xlabel("x")
ax2.set_ylabel("u(x)")
ax2.set_title("Diffusion: Initial vs Final")
ax2.legend()
ax2.grid(True)

# dt choices over time
ax3 = plt.subplot(2, 3, 3)
ax3.plot(dt_choices, marker='o', markersize=3, linewidth=1)
ax3.set_xlabel("Rollout Step")
ax3.set_ylabel("dt (chosen)")
ax3.set_title("Agent's dt Choices")
ax3.set_ylim([0.5, 8.5])
ax3.grid(True)

# dt histogram
ax4 = plt.subplot(2, 3, 4)
dt_counts = {dt: dt_choices.count(dt) for dt in agent.dt_choices}
ax4.bar(dt_counts.keys(), dt_counts.values(), edgecolor='black', alpha=0.7)
ax4.set_xlabel("dt value")
ax4.set_ylabel("Frequency")
ax4.set_title("Distribution of dt Choices (Test)")
ax4.set_xticks(agent.dt_choices)
ax4.grid(True, alpha=0.3, axis='y')

# Field evolution at key times
ax5 = plt.subplot(2, 3, 5)
for i in [0, len(field_states)//4, len(field_states)//2, 3*len(field_states)//4, -1]:
    if i < len(field_states):
        field_val = field_states[i].cpu().numpy() if isinstance(field_states[i], torch.Tensor) else field_states[i]
        ax5.plot(x, field_val, alpha=0.6, label=f"Step {i}")
ax5.set_xlabel("x")
ax5.set_ylabel("u(x)")
ax5.set_title("Field Evolution During Rollout")
ax5.legend()
ax5.grid(True)

# Episode stats
ax6 = plt.subplot(2, 3, 6)
ax6.axis('off')
# Compute dt distribution
dt_dist_test = {dt: dt_choices.count(dt) for dt in agent.dt_choices}
dt_dist_pct = {dt: 100*dt_choices.count(dt)/len(dt_choices) for dt in agent.dt_choices}

stats_text = f"""
Training Statistics:
• Final Avg Reward: {episode_rewards[-1]:.2f}
• Final Avg dt: {episode_avg_dt[-1]:.2f}
• Final Avg Error: {episode_avg_error[-1]:.2e}
• Final Entropy: {episode_entropy[-1]:.3f}
• Episodes: {n_episodes}

Learned dt Strategy (Test):
• Mean dt: {np.mean(dt_choices):.2f}
• Std dt: {np.std(dt_choices):.2f}
• Range: [{np.min(dt_choices)}, {np.max(dt_choices)}]
• Total rollout steps: {len(dt_choices)}

dt Distribution (%):
{chr(10).join([f'  dt={dt}: {dt_dist_pct[dt]:.1f}%' for dt in agent.dt_choices])}
"""
ax6.text(0.1, 0.5, stats_text, fontsize=10, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

print("\nTraining complete!")


In [ ]:
# ============================================================
# Additional Diagnostic Plots: Training Evolution & Adaptation Analysis
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# 1. Reward, dt, error, and entropy over training
ax1 = axes[0, 0]
ax1_twin = ax1.twinx()
ax1.plot(episode_rewards, 'b-', label='Reward', linewidth=2)
ax1_twin.plot(episode_avg_dt, 'r-', label='Avg dt', linewidth=2)
ax1.set_xlabel("Episode")
ax1.set_ylabel("Reward", color='b')
ax1_twin.set_ylabel("Avg dt", color='r')
ax1.set_title("Reward and Avg dt Over Training")
ax1.tick_params(axis='y', labelcolor='b')
ax1_twin.tick_params(axis='y', labelcolor='r')
ax1.grid(True, alpha=0.3)
ax1.legend(loc='upper left')
ax1_twin.legend(loc='upper right')

# 2. Error and entropy over training
ax2 = axes[0, 1]
ax2_twin = ax2.twinx()
ax2.semilogy(episode_avg_error, 'g-', label='Error', linewidth=2)
ax2_twin.plot(episode_entropy, 'm-', label='Entropy', linewidth=2)
ax2.set_xlabel("Episode")
ax2.set_ylabel("Error (log scale)", color='g')
ax2_twin.set_ylabel("Policy Entropy", color='m')
ax2.set_title("Error and Policy Entropy Over Training")
ax2.tick_params(axis='y', labelcolor='g')
ax2_twin.tick_params(axis='y', labelcolor='m')
ax2.grid(True, alpha=0.3)
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')

# 3. dt Distribution Evolution (stacked area)
ax3 = axes[1, 0]
dt_choices_array = np.array([agent.dt_choices])
dt_dist_matrix = np.zeros((len(episode_dt_distribution), len(agent.dt_choices)))
for i, dist in enumerate(episode_dt_distribution):
    for j, dt in enumerate(agent.dt_choices):
        dt_dist_matrix[i, j] = dist.get(dt, 0)

# Normalize to percentages
dt_dist_pct = dt_dist_matrix / (dt_dist_matrix.sum(axis=1, keepdims=True) + 1e-8) * 100

ax3.stackplot(range(len(episode_dt_distribution)), 
              *[dt_dist_pct[:, i] for i in range(len(agent.dt_choices))],
              labels=[f'dt={dt}' for dt in agent.dt_choices],
              alpha=0.7)
ax3.set_xlabel("Episode")
ax3.set_ylabel("Percentage of dt Choices (%)")
ax3.set_title("dt Distribution Evolution Over Training")
ax3.legend(loc='upper right')
ax3.grid(True, alpha=0.3)
ax3.set_ylim([0, 100])

# 4. dt vs Gradient Norm (adaptation analysis)
ax4 = axes[1, 1]
if 'state_features_list' in globals() and len(state_features_list) > 0 and 'dt_choices' in globals() and len(dt_choices) > 0:
    state_features_array = np.array(state_features_list)
    grad_norms = state_features_array[:, 2]  # Gradient norm is 3rd feature (index 2)
    
    # Create bins for gradient norm
    n_bins = 20
    grad_bins = np.linspace(grad_norms.min(), grad_norms.max(), n_bins+1)
    bin_centers = (grad_bins[:-1] + grad_bins[1:]) / 2
    
    # Compute mean dt for each bin
    mean_dt_per_bin = []
    for i in range(n_bins):
        mask = (grad_norms >= grad_bins[i]) & (grad_norms < grad_bins[i+1])
        if mask.sum() > 0:
            mean_dt_per_bin.append(np.mean([dt_choices[j] for j in range(len(dt_choices)) if mask[j]]))
        else:
            mean_dt_per_bin.append(np.nan)
    
    ax4.plot(bin_centers, mean_dt_per_bin, 'o-', linewidth=2, markersize=6)
    ax4.set_xlabel("Gradient Norm (normalized)")
    ax4.set_ylabel("Mean dt chosen")
    ax4.set_title("Adaptation: dt vs Gradient Norm")
    ax4.grid(True, alpha=0.3)
    ax4.axhline(y=np.mean(dt_choices), color='r', linestyle='--', 
                label=f'Overall mean dt={np.mean(dt_choices):.2f}')
    ax4.legend()

# 5. Critic Loss over time
ax5 = axes[2, 0]
if len(episode_critic_loss) > 0:
    ax5.plot(episode_critic_loss, 'purple', linewidth=2)
    ax5.set_xlabel("Episode")
    ax5.set_ylabel("Critic Loss")
    ax5.set_title("Critic Loss Over Training")
    ax5.grid(True, alpha=0.3)
    ax5.set_yscale('log')
else:
    ax5.text(0.5, 0.5, 'Critic Loss not tracked\n(Run training cell first)', 
             ha='center', va='center', transform=ax5.transAxes, fontsize=12)
    ax5.set_title("Critic Loss Over Training")
    ax5.axis('off')

# 6. dt Choice Frequency Heatmap (over episodes)
ax6 = axes[2, 1]
# Create heatmap of dt choices over episodes (sample every 10 episodes)
sample_episodes = list(range(0, len(episode_dt_distribution), max(1, len(episode_dt_distribution)//20)))
if len(sample_episodes) > 0:
    heatmap_data = np.zeros((len(sample_episodes), len(agent.dt_choices)))
    for i, ep_idx in enumerate(sample_episodes):
        dist = episode_dt_distribution[ep_idx]
        total = sum(dist.values())
        for j, dt in enumerate(agent.dt_choices):
            heatmap_data[i, j] = dist.get(dt, 0) / (total + 1e-8) * 100
    
    im = ax6.imshow(heatmap_data.T, aspect='auto', cmap='viridis', interpolation='nearest')
    ax6.set_xlabel("Episode (sampled)")
    ax6.set_ylabel("dt value")
    ax6.set_yticks(range(len(agent.dt_choices)))
    ax6.set_yticklabels(agent.dt_choices)
    ax6.set_title("dt Choice Frequency Heatmap Over Training")
    plt.colorbar(im, ax=ax6, label="Percentage (%)")
else:
    ax6.text(0.5, 0.5, 'Not enough data', ha='center', va='center', 
             transform=ax6.transAxes, fontsize=12)
    ax6.axis('off')

plt.tight_layout()
plt.show()

# Print adaptation analysis
print("\n" + "="*60)
print("ADAPTATION ANALYSIS")
print("="*60)
if 'state_features_list' in globals() and len(state_features_list) > 0 and 'dt_choices' in globals() and len(dt_choices) > 0:
    state_features_array = np.array(state_features_list)
    grad_norms = state_features_array[:, 2]
    
    # Split into low and high gradient regions
    grad_median = np.median(grad_norms)
    low_grad_mask = grad_norms < grad_median
    high_grad_mask = grad_norms >= grad_median
    
    low_grad_dt = [dt_choices[i] for i in range(len(dt_choices)) if low_grad_mask[i]]
    high_grad_dt = [dt_choices[i] for i in range(len(dt_choices)) if high_grad_mask[i]]
    
    print(f"Low gradient regions (grad < {grad_median:.3f}):")
    print(f"  Mean dt: {np.mean(low_grad_dt):.2f}, Std: {np.std(low_grad_dt):.2f}")
    print(f"  Count: {len(low_grad_dt)}")
    
    print(f"\nHigh gradient regions (grad >= {grad_median:.3f}):")
    print(f"  Mean dt: {np.mean(high_grad_dt):.2f}, Std: {np.std(high_grad_dt):.2f}")
    print(f"  Count: {len(high_grad_dt)}")
    
    adaptation_score = abs(np.mean(low_grad_dt) - np.mean(high_grad_dt))
    print(f"\nAdaptation Score: {adaptation_score:.2f}")
    if adaptation_score > 1.0:
        print("  ✓ Agent IS adapting (uses different dt based on gradient)")
    else:
        print("  ✗ Agent NOT adapting well (similar dt regardless of gradient)")
    
    # Check if agent uses dt=9
    dt9_count = dt_choices.count(9)
    dt9_pct = 100 * dt9_count / len(dt_choices) if len(dt_choices) > 0 else 0
    print(f"\ndt=9 Usage: {dt9_count} times ({dt9_pct:.1f}%)")
    if dt9_count > 0:
        print("  ✓ Agent uses dt=9 (generalizes beyond training range)")
    else:
        print("  ✗ Agent never uses dt=9 (may be stuck at max trained dt=8)")

print("="*60)


In [ ]:
# ============================================================
# Plot: Spatial Map of dt choices + State Features Analysis
# ============================================================

# Build a (steps × Nx) matrix where each row is filled with the dt chosen at that step
if 'dt_choices' in globals() and len(dt_choices) > 0:
    dt_map = np.zeros((len(dt_choices), Nx))
    
    for i, dt in enumerate(dt_choices):
        dt_map[i, :] = dt  # same dt repeated across spatial dimension
else:
    print("Warning: dt_choices not found. Please run Cell 6 first.")
    dt_map = np.zeros((1, Nx))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# dt choices map
ax1 = axes[0, 0]
if 'dt_choices' in globals() and len(dt_choices) > 0:
    im1 = ax1.imshow(dt_map, aspect='auto', cmap='viridis',
               extent=[0, 1, len(dt_choices), 0])
    ax1.set_xlabel("Spatial location x")
    ax1.set_ylabel("Rollout step")
    ax1.set_title("Spatial-Temporal Map of dt Choices")
    plt.colorbar(im1, ax=ax1, label="dt multiplier")
else:
    ax1.text(0.5, 0.5, 'No data available\n(Run Cell 6 first)', 
             ha='center', va='center', transform=ax1.transAxes, fontsize=12)
    ax1.set_title("Spatial-Temporal Map of dt Choices")
    ax1.axis('off')

# State features over time
ax2 = axes[0, 1]
if 'state_features_list' in globals() and len(state_features_list) > 0:
    state_features_array = np.array(state_features_list)
    ax2.plot(state_features_array[:, 0], label='Mean (norm)', alpha=0.7)
    ax2.plot(state_features_array[:, 1], label='Std (norm)', alpha=0.7)
    ax2.plot(state_features_array[:, 2], label='Grad norm (norm)', alpha=0.7)
    ax2.set_xlabel("Rollout step")
    ax2.set_ylabel("Feature value")
    ax2.set_title("State Features Over Time")
    ax2.legend()
    ax2.grid(True, alpha=0.3)
else:
    ax2.text(0.5, 0.5, 'No data available\n(Run Cell 6 first)', 
             ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title("State Features Over Time")
    ax2.axis('off')

# dt vs state features
ax3 = axes[1, 0]
if 'state_features_list' in globals() and len(state_features_list) > 0 and 'dt_choices' in globals() and len(dt_choices) > 0:
    state_features_array = np.array(state_features_list)
    scatter = ax3.scatter(state_features_array[:, 2], dt_choices, 
                         c=state_features_array[:, 1], cmap='coolwarm', alpha=0.6)
    ax3.set_xlabel("Gradient Norm (normalized)")
    ax3.set_ylabel("dt chosen")
    ax3.set_title("dt Choice vs Gradient Norm")
    ax3.grid(True, alpha=0.3)
    plt.colorbar(scatter, ax=ax3, label="Std (normalized)")
else:
    ax3.text(0.5, 0.5, 'No data available\n(Run Cell 6 first)', 
             ha='center', va='center', transform=ax3.transAxes, fontsize=12)
    ax3.set_title("dt Choice vs Gradient Norm")
    ax3.axis('off')

# Reward and dt evolution during training
ax4 = axes[1, 1]
if 'episode_rewards' in globals() and len(episode_rewards) > 0 and 'episode_avg_dt' in globals() and len(episode_avg_dt) > 0:
    ax4_twin = ax4.twinx()
    ax4.plot(episode_rewards, 'b-', label='Reward', linewidth=2)
    ax4_twin.plot(episode_avg_dt, 'r-', label='Avg dt', linewidth=2)
    ax4.set_xlabel("Episode")
    ax4.set_ylabel("Reward", color='b')
    ax4_twin.set_ylabel("Avg dt", color='r')
    ax4.set_title("Training Progress")
    ax4.tick_params(axis='y', labelcolor='b')
    ax4_twin.tick_params(axis='y', labelcolor='r')
    ax4.grid(True, alpha=0.3)
else:
    ax4.text(0.5, 0.5, 'No training data available\n(Run Cell 4 first)', 
             ha='center', va='center', transform=ax4.transAxes, fontsize=12)
    ax4.set_title("Training Progress")
    ax4.axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("SUMMARY OF CHANGES MADE")
print("="*60)
print("""
1. STATE REPRESENTATION:
   OLD: Full 256-dim field → CNN encoder
   NEW: 3 summary statistics [mean(u), std(u), gradient_norm(u)]
   WHY: Simpler, more interpretable, faster training

2. ACTION SPACE:
   OLD: Continuous [1, 8] → converted to int (breaks gradients)
   NEW: Discrete {1, 2, 4, 8} (proper categorical distribution)
   WHY: Cleaner policy gradients, no discontinuity issues

3. REWARD FUNCTION:
   OLD: reward = 1/(1+error) - 0.01*(dt/8)  [saturated at ~0.998]
   NEW: reward = -log(error + ε) + 0.1*dt  [log scale, speed bonus]
   WHY: Better gradient signal, proper accuracy-speed tradeoff

4. NETWORK ARCHITECTURE:
   OLD: CNN (Conv1d) → FC → Actor/Critic (128 hidden)
   NEW: Simple MLP (64 hidden) → Actor/Critic
   WHY: Simpler, faster, easier to debug

5. TRAINING:
   - Added tracking of avg dt and avg error per episode
   - Better diagnostics to see if agent is learning
""")
print("="*60)


In [ ]:
# ============================================================
# Export High-Quality Figures: dt Distribution & dt Choices Over Time
# ============================================================
# Replotting with the same style as diffusion notebook for publication quality

import os
os.makedirs('RL_figures', exist_ok=True)  # Create directory for RL figures

# High-quality plotting setup (same as diffusion notebook)
plt.rcParams['figure.dpi'] = 300  # DPI for display
plt.rcParams['savefig.format'] = 'pdf'  # PDF format for vector graphics
plt.rcParams['savefig.pad_inches'] = 0.1
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 14
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif']
plt.rcParams['axes.linewidth'] = 1.2
plt.rcParams['grid.linewidth'] = 0.8
plt.rcParams['lines.linewidth'] = 2.5
plt.rcParams['lines.markersize'] = 6
plt.rcParams['lines.antialiased'] = True
plt.rcParams['patch.antialiased'] = True
plt.rcParams['text.antialiased'] = True

# Check if dt_choices exists (from Cell 6)
if 'dt_choices' not in globals() or len(dt_choices) == 0:
    print("Warning: dt_choices not found. Please run Cell 6 first to generate the data.")
else:
    # ============================================================
    # Figure 1: Distribution of dt Choices (Test) - Bar Chart
    # ============================================================
    fig1, ax1 = plt.subplots(figsize=(10, 6), dpi=300)
    
    # Compute dt counts
    dt_counts = {dt: dt_choices.count(dt) for dt in agent.dt_choices}
    
    # Create bar chart with nice colors
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#BC4749']
    bars = ax1.bar(dt_counts.keys(), dt_counts.values(), 
                   color=colors[:len(dt_counts)], 
                   edgecolor='black', 
                   linewidth=1.5,
                   alpha=0.8,
                   width=0.6)
    
    ax1.set_xlabel("dt value", fontsize=12, fontweight='bold')
    ax1.set_ylabel("Frequency", fontsize=12, fontweight='bold')
    ax1.set_title("Distribution of dt Choices (Test)", fontsize=14, fontweight='bold', pad=15)
    ax1.set_xticks(list(dt_counts.keys()))
    ax1.grid(True, alpha=0.3, linestyle='--', linewidth=0.8, axis='y')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    
    # Add value labels on top of bars
    for dt, count in dt_counts.items():
        ax1.text(dt, count, str(count), ha='center', va='bottom', 
                fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    fig1.savefig('RL_figures/qlearning_dt_distribution.pdf', bbox_inches='tight', 
                 facecolor='white', edgecolor='none')
    print("✓ Saved: RL_figures/qlearning_dt_distribution.pdf")
    plt.show()
    
    # ============================================================
    # Figure 2: dt Choices Over Time - Line Plot
    # ============================================================
    fig2, ax2 = plt.subplots(figsize=(12, 6), dpi=300)
    
    # Plot dt choices over rollout steps
    ax2.plot(dt_choices, linewidth=2.5, color='#2E86AB', marker='o', 
             markersize=3, markevery=max(1, len(dt_choices)//100), 
             alpha=0.8, label='dt choices')
    
    ax2.set_xlabel("Rollout Step", fontsize=12, fontweight='bold')
    ax2.set_ylabel("dt (chosen)", fontsize=12, fontweight='bold')
    ax2.set_title("Agent's dt Choices Over Time", fontsize=14, fontweight='bold', pad=15)
    ax2.set_ylim([0.5, max(agent.dt_choices) + 0.5])
    ax2.set_yticks(agent.dt_choices)
    ax2.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
    ax2.legend(frameon=True, fancybox=True, shadow=True, loc='best')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    
    # Add horizontal line for mean dt
    mean_dt = np.mean(dt_choices)
    ax2.axhline(y=mean_dt, color='#C73E1D', linestyle='--', linewidth=2, 
               alpha=0.7, label=f'Mean dt = {mean_dt:.2f}')
    ax2.legend(frameon=True, fancybox=True, shadow=True, loc='best')
    
    plt.tight_layout()
    fig2.savefig('RL_figures/qlearning_dt_choices_over_time.pdf', bbox_inches='tight', 
                 facecolor='white', edgecolor='none')
    print("✓ Saved: RL_figures/qlearning_dt_choices_over_time.pdf")
    plt.show()
    
    # ============================================================
    # Figure 3: Gradient Norm vs dt Choices - Improved Visualization
    # ============================================================
    if 'state_features_list' in globals() and len(state_features_list) > 0:
        fig3, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=300)
        
        # Extract gradient norms (3rd feature, index 2)
        state_features_array = np.array(state_features_list)
        grad_norms = state_features_array[:, 2]
        
        # Left plot: Scatter with jitter for better visibility
        ax3a = axes[0]
        dt_unique = sorted(set(dt_choices))
        colors_map = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#6A994E', '#BC4749']
        dt_color_dict = {dt: colors_map[i % len(colors_map)] for i, dt in enumerate(dt_unique)}
        
        # Add small jitter to y-axis for better visibility
        np.random.seed(42)  # For reproducibility
        y_jitter = np.array(dt_choices) + np.random.normal(0, 0.1, len(dt_choices))
        
        # Plot each dt value with different color
        for dt_val in dt_unique:
            mask = np.array(dt_choices) == dt_val
            ax3a.scatter(grad_norms[mask], y_jitter[mask], 
                        c=dt_color_dict[dt_val], label=f'dt={dt_val}', 
                        alpha=0.5, s=20, edgecolors='black', linewidth=0.3)
        
        ax3a.set_xlabel("Gradient Norm (normalized)", fontsize=12, fontweight='bold')
        ax3a.set_ylabel("dt (chosen)", fontsize=12, fontweight='bold')
        ax3a.set_title("Gradient Norm vs dt Choices (Scatter)", fontsize=13, fontweight='bold', pad=15)
        ax3a.set_yticks(agent.dt_choices)
        ax3a.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
        ax3a.spines['top'].set_visible(False)
        ax3a.spines['right'].set_visible(False)
        
        # Add mean line for reference
        mean_grad = np.mean(grad_norms)
        ax3a.axvline(x=mean_grad, color='#C73E1D', linestyle='--', linewidth=2, 
                     alpha=0.7, label=f'Mean = {mean_grad:.4f}')
        ax3a.legend(frameon=True, fancybox=True, shadow=True, loc='best', ncol=2, fontsize=9)
        
        # Right plot: Binned mean dt per gradient bin (more interpretable)
        ax3b = axes[1]
        
        # Create bins for gradient norm
        n_bins = 20
        grad_bins = np.linspace(grad_norms.min(), grad_norms.max(), n_bins+1)
        bin_centers = (grad_bins[:-1] + grad_bins[1:]) / 2
        
        # Compute mean dt for each bin
        mean_dt_per_bin = []
        std_dt_per_bin = []
        counts_per_bin = []
        for i in range(n_bins):
            mask = (grad_norms >= grad_bins[i]) & (grad_norms < grad_bins[i+1])
            if i == n_bins - 1:  # Include last point
                mask = (grad_norms >= grad_bins[i]) & (grad_norms <= grad_bins[i+1])
            if mask.sum() > 0:
                dt_in_bin = [dt_choices[j] for j in range(len(dt_choices)) if mask[j]]
                mean_dt_per_bin.append(np.mean(dt_in_bin))
                std_dt_per_bin.append(np.std(dt_in_bin))
                counts_per_bin.append(len(dt_in_bin))
            else:
                mean_dt_per_bin.append(np.nan)
                std_dt_per_bin.append(np.nan)
                counts_per_bin.append(0)
        
        # Plot mean dt with error bars
        valid_mask = ~np.isnan(mean_dt_per_bin)
        ax3b.plot(bin_centers[valid_mask], np.array(mean_dt_per_bin)[valid_mask], 
                  'o-', color='#2E86AB', linewidth=2.5, markersize=6, 
                  label='Mean dt per gradient bin')
        ax3b.fill_between(bin_centers[valid_mask], 
                          np.array(mean_dt_per_bin)[valid_mask] - np.array(std_dt_per_bin)[valid_mask],
                          np.array(mean_dt_per_bin)[valid_mask] + np.array(std_dt_per_bin)[valid_mask],
                          alpha=0.3, color='#2E86AB', label='±1 std')
        
        ax3b.set_xlabel("Gradient Norm (normalized)", fontsize=12, fontweight='bold')
        ax3b.set_ylabel("Mean dt (chosen)", fontsize=12, fontweight='bold')
        ax3b.set_title("Mean dt vs Gradient Norm (Binned)", fontsize=13, fontweight='bold', pad=15)
        ax3b.grid(True, alpha=0.3, linestyle='--', linewidth=0.8)
        ax3b.spines['top'].set_visible(False)
        ax3b.spines['right'].set_visible(False)
        ax3b.axvline(x=mean_grad, color='#C73E1D', linestyle='--', linewidth=2, 
                     alpha=0.7, label=f'Mean grad = {mean_grad:.4f}')
        ax3b.axhline(y=np.mean(dt_choices), color='#F18F01', linestyle='--', linewidth=2, 
                     alpha=0.7, label=f'Overall mean dt = {np.mean(dt_choices):.2f}')
        ax3b.legend(frameon=True, fancybox=True, shadow=True, loc='best', fontsize=9)
        
        plt.tight_layout()
        fig3.savefig('RL_figures/qlearning_gradient_vs_dt.pdf', bbox_inches='tight', 
                     facecolor='white', edgecolor='none')
        print("✓ Saved: RL_figures/qlearning_gradient_vs_dt.pdf")
        
        # Print correlation statistics
        correlation = np.corrcoef(grad_norms, dt_choices)[0, 1]
        print(f"\nCorrelation between gradient norm and dt: {correlation:.4f}")
        if abs(correlation) < 0.1:
            print("  → Very weak correlation - agent not adapting based on gradient")
        elif abs(correlation) < 0.3:
            print("  → Weak correlation - minimal adaptation")
        elif abs(correlation) < 0.5:
            print("  → Moderate correlation - some adaptation")
        else:
            print("  → Strong correlation - good adaptation!")
        
        plt.show()
    else:
        print("Warning: state_features_list not found. Skipping gradient vs dt plot.")
        print("         Please run Cell 6 first to generate the data.")
    
    print("\n" + "="*60)
    print("High-quality PDF figures exported successfully!")
    print("="*60)
